# 13 — Real-data ResNet50 training with MILK10k
**Use a T4 GPU.** Run all code cells in order. This notebook is self-contained and needs no variables from other notebooks.

Continue from the completed notebook 09 ResNet50 checkpoint (`exp07_2cc989221ca4`), adding only the 2,101 MILK training photographs prepared in notebook 12. Expected combined training size: **4,291**, including **447 SCC**. Do not regenerate splits.

Selection is fixed in advance: **macro F1 on the original 390-image validation set**. Report MILK's separate 451-image validation results as a secondary outcome; do not pool them to conceal differences between sources. Keep the pre-training checkpoint if no epoch improves the original validation macro F1. Both test partitions are reserved; their manifest checksums are verified, but their image bytes and predictions are not loaded.

This is a real-data experiment, not synthetic augmentation. It changes data volume, class-weight values and starting checkpoint; it does not isolate the causal effect of adding MILK. A future synthetic comparison must use the same real-data training configuration and fixed real validation partitions. MILK skin-tone classes are not Fitzpatrick types, and its sparse darker-SCC representation cannot establish darker-skin reliability.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image, ImageOps
import gc, hashlib, io, json, shutil, time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix

ROOT = Path('/content/drive/MyDrive/cnn-skin-cancer-classification')
MILK_ROOT = ROOT / 'Datasets' / 'MILK10k' / 'Prepared' / '20261104_175741_606266'
EXTERNAL_MANIFEST = ROOT / 'Reports' / 'External_Data_Preparation' / 'external_training_manifest.csv'
SOURCE_RUN = 'exp07_2cc989221ca4'
SOURCE_MODEL_ROOT = ROOT / 'Models' / 'Transfer_Learning' / SOURCE_RUN / 'ResNet50'
CLASS_NAMES = ['BCC', 'Melanoma', 'SCC']
CLASS_TO_INDEX = {label:i for i,label in enumerate(CLASS_NAMES)}
SEED=42; IMAGE_SIZE=(224,224); BATCH_SIZE=16
HEAD_EPOCHS=6; FINE_TUNE_EPOCHS=20
HEAD_LR=1e-4; FINE_LR=1e-5

print('TensorFlow:',tf.__version__)
print('Keras:',getattr(tf.keras,'__version__','bundled'))
gpus=tf.config.list_physical_devices('GPU')
print('Available GPUs:',gpus)
if not gpus: raise RuntimeError('Select Runtime → Change runtime type → T4 GPU, then rerun from the top.')
for gpu in gpus:
    try: tf.config.experimental.set_memory_growth(gpu,True)
    except RuntimeError: pass
tf.keras.utils.set_random_seed(SEED)

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''): h.update(block)
    return h.hexdigest()

def write_json(path,value):
    path=Path(path); temporary=path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value,indent=2),encoding='utf-8'); temporary.replace(path)

def validate_frame(frame, split, name):
    required={'dataset','image_id','image_path','class_label','label_index','split'}
    if not required.issubset(frame.columns): raise ValueError(name+': required manifest columns missing.')
    if frame.empty or not frame.split.eq(split).all(): raise ValueError(name+': incorrect or empty split.')
    if set(frame.class_label)!=set(CLASS_NAMES): raise ValueError(name+': expected all three classes.')
    if not frame.class_label.map(CLASS_TO_INDEX).eq(pd.to_numeric(frame.label_index,errors='raise')).all():
        raise ValueError(name+': class order differs.')
    if frame[['dataset','image_id']].duplicated().any(): raise ValueError(name+': duplicate identifiers.')
    if frame.image_path.isna().any(): raise ValueError(name+': missing image paths.')
    return frame.copy()

if not MILK_ROOT.is_dir(): raise FileNotFoundError('Notebook 12 prepared folder missing: '+str(MILK_ROOT))
preparation=json.loads((MILK_ROOT/'preparation_settings.json').read_text())
original_hashes=preparation['original_split_sha256']
milk_hashes=preparation['split_manifest_sha256']
# Verify held-out manifest checksums without reading test image data or predictions.
for split in ['train','validation','test']:
    original_path=ROOT/'Datasets'/'Splits'/f'{split}_metadata.csv'
    milk_path=MILK_ROOT/f'{split}_metadata.csv'
    if sha256_file(original_path)!=original_hashes[split]: raise ValueError('Original split changed: '+split)
    if sha256_file(milk_path)!=milk_hashes[split]: raise ValueError('MILK split changed: '+split)
if sha256_file(EXTERNAL_MANIFEST)!=preparation['external_training_manifest_sha256']:
    raise ValueError('Fitzpatrick training manifest changed since preparation.')
if not preparation.get('paired_modalities_screened') or preparation['image_errors']!=0:
    raise ValueError('Notebook 12 image screening is incomplete.')

original_train=validate_frame(pd.read_csv(ROOT/'Datasets'/'Splits'/'train_metadata.csv'),'train','original train')
original_val=validate_frame(pd.read_csv(ROOT/'Datasets'/'Splits'/'validation_metadata.csv'),'validation','original validation')
external_train=validate_frame(pd.read_csv(EXTERNAL_MANIFEST),'train','Fitzpatrick training')
milk_train=validate_frame(pd.read_csv(MILK_ROOT/'train_metadata.csv'),'train','MILK training')
milk_val=validate_frame(pd.read_csv(MILK_ROOT/'validation_metadata.csv'),'validation','MILK validation')
if not external_train.dataset.eq('Fitzpatrick17k').all(): raise ValueError('Unexpected external training source.')
if not milk_train.dataset.eq('MILK10k').all() or not milk_val.dataset.eq('MILK10k').all(): raise ValueError('Unexpected MILK source.')
train_frame=pd.concat([original_train,external_train,milk_train],ignore_index=True)
validate_frame(train_frame,'train','combined training')
val_frames={'original':original_val,'MILK10k':milk_val}
all_fit_paths=set(train_frame.image_path)
for name,frame in val_frames.items():
    if all_fit_paths & set(frame.image_path): raise ValueError('Training/validation image-path overlap: '+name)
    if set(zip(train_frame.dataset,train_frame.image_id)) & set(zip(frame.dataset,frame.image_id)):
        raise ValueError('Training/validation identifier overlap: '+name)
if set(milk_train.lesion_id)&set(milk_val.lesion_id): raise ValueError('MILK lesion overlap.')

source_metadata=json.loads((SOURCE_MODEL_ROOT/'metadata.json').read_text())
if not source_metadata.get('training_complete') or source_metadata['run_id']!=SOURCE_RUN:
    raise ValueError('Expected completed notebook 09 checkpoint.')
if source_metadata['class_names']!=CLASS_NAMES or source_metadata['image_size']!=list(IMAGE_SIZE) or source_metadata['preprocessing']!='resnet50':
    raise ValueError('Source model class order, resolution or preprocessing mismatch.')
if source_metadata['manifest_hashes']!=original_hashes: raise ValueError('Source checkpoint used different original splits.')
if source_metadata['experiment_settings']['external_manifest_sha256']!=sha256_file(EXTERNAL_MANIFEST):
    raise ValueError('Source checkpoint used different external training data.')
SOURCE_MODEL=SOURCE_MODEL_ROOT/'best.keras'
if sha256_file(SOURCE_MODEL)!=source_metadata['model_sha256']: raise ValueError('Source checkpoint checksum mismatch.')

train_counts=train_frame.label_index.value_counts().reindex(range(3))
ALPHA=[float(len(train_frame)/(3*train_counts[i])) for i in range(3)]
SETTINGS={'experiment':'experiment_08_real_MILK_warm_start','source_run':SOURCE_RUN,
          'source_model_sha256':source_metadata['model_sha256'],'seed':SEED,
          'original_split_sha256':original_hashes,'milk_split_sha256':milk_hashes,
          'external_training_manifest_sha256':sha256_file(EXTERNAL_MANIFEST),
          'milk_preparation_settings_sha256':sha256_file(MILK_ROOT/'preparation_settings.json'),
          'class_names':CLASS_NAMES,'image_size':list(IMAGE_SIZE),'batch_size':BATCH_SIZE,
          'head_epochs':HEAD_EPOCHS,'fine_tune_epochs':FINE_TUNE_EPOCHS,'head_lr':HEAD_LR,'fine_lr':FINE_LR,
          'loss':'weighted categorical crossentropy (focal gamma 0)','alpha':ALPHA,
          'selection_metric':'original_validation_macro_f1','secondary_metric':'MILK_validation_macro_f1',
          'synthetic_images_used':False,'test_inference_performed':False}
RUN_ID='exp08_'+hashlib.sha256(json.dumps(SETTINGS,sort_keys=True).encode()).hexdigest()[:12]
RUN_MODELS=ROOT/'Models'/'Transfer_Learning'/RUN_ID/'ResNet50'
RUN_REPORTS=ROOT/'Reports'/'Training'/RUN_ID/'ResNet50'
LOCAL_ROOT=Path('/content/skin_cancer_milk_training_images')
for p in [RUN_MODELS,RUN_REPORTS,LOCAL_ROOT]:p.mkdir(parents=True,exist_ok=True)
# Preserve interrupted work and never overwrite a completed run.
existing_metadata=RUN_MODELS/'metadata.json'
if existing_metadata.is_file():
    old=json.loads(existing_metadata.read_text())
    if old.get('training_complete'):
        raise RuntimeError('This run is already complete. Read its saved reports; do not retrain or overwrite it: '+str(RUN_REPORTS))
if (RUN_MODELS/'best.keras').exists() or (RUN_REPORTS/'training_history.csv').exists():
    stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    RUN_MODELS.rename(RUN_MODELS.with_name('ResNet50_interrupted_'+stamp))
    RUN_REPORTS.rename(RUN_REPORTS.with_name('ResNet50_interrupted_'+stamp))
    RUN_MODELS.mkdir();RUN_REPORTS.mkdir()
write_json(RUN_REPORTS/'training_settings.json',SETTINGS)
train_frame.to_csv(RUN_REPORTS/'training_manifest.csv',index=False)
for name,frame in val_frames.items(): frame.to_csv(RUN_REPORTS/f'{name}_validation_manifest.csv',index=False)
print('Training run:',RUN_ID)
print('Training images:',len(train_frame),'SCC:',int(train_counts[2]))
print('Original validation:',len(original_val),'MILK validation:',len(milk_val))
print('Training-only class weights:',ALPHA)
display(pd.crosstab(train_frame.dataset,train_frame.class_label))
print('No test images or predictions loaded.')

## Verify and copy training/validation images locally
Match original notebook 03 pixel hashes, notebook 09 Fitzpatrick byte hashes and notebook 12 MILK byte hashes. Training inputs use original orientation, TensorFlow RGB decode and bilinear resize, matching the existing model. Reference pixel hashes are used for integrity only. Augmentation is training-only. Read images from local disk during fitting; avoid storing thousands of float images in RAM.

In [ ]:
def reference_digest(raw,dataset):
    if dataset=='Fitzpatrick17k' or dataset=='MILK10k':return hashlib.sha256(raw).hexdigest()
    if dataset not in {'HAM10000','PAD-UFES-20'}:raise ValueError('Unknown source hash scheme: '+str(dataset))
    with Image.open(io.BytesIO(raw)) as image:
        rgb=ImageOps.exif_transpose(image).convert('RGB')
        h=hashlib.sha256();h.update(f'{rgb.width}x{rgb.height}:RGB:'.encode());h.update(rgb.tobytes())
        return h.hexdigest()

for split,frame in [('train',train_frame),*[(name+'_validation',frame) for name,frame in val_frames.items()]]:
    paths=[]
    for position,row in enumerate(frame.to_dict('records'),1):
        source=Path(row['image_path'])
        if not source.is_file():raise FileNotFoundError(source)
        raw=source.read_bytes()
        if reference_digest(raw,row['dataset'])!=str(row['image_hash']):raise ValueError('Source image integrity mismatch: '+str(source))
        with Image.open(io.BytesIO(raw)) as image:
            if image.width*image.height>20_000_000:raise ValueError('Pixel limit exceeded: '+str(source))
        if source.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp'}:raise ValueError('Unsupported format: '+str(source))
        byte_hash=hashlib.sha256(raw).hexdigest()
        destination=LOCAL_ROOT/(byte_hash+source.suffix.lower())
        if destination.is_file():
            if sha256_file(destination)!=byte_hash:raise ValueError('Cached local bytes differ: '+str(destination))
        else:
            temporary=destination.with_suffix(destination.suffix+'.part');temporary.write_bytes(raw);temporary.replace(destination)
        paths.append(str(destination))
        if position%500==0:print(split,position,'/',len(frame),'verified and copied',flush=True)
    frame['local_path']=paths
    print(split,'local images ready:',len(paths))

def decode_image(path,label):
    image=tf.io.decode_image(tf.io.read_file(path),channels=3,expand_animations=False)
    image.set_shape([None,None,3])
    return tf.image.resize(tf.cast(image,tf.float32),IMAGE_SIZE,method='bilinear'),label

def image_dataset(frame,training=False):
    ds=tf.data.Dataset.from_tensor_slices((frame.local_path.to_numpy(dtype=str),frame.label_index.to_numpy(dtype=np.int32)))
    if training:ds=ds.shuffle(len(frame),seed=SEED,reshuffle_each_iteration=True)
    ds=ds.map(decode_image,num_parallel_calls=4).batch(BATCH_SIZE)
    if training:
        augmentation=tf.keras.Sequential([
            tf.keras.layers.RandomFlip('horizontal_and_vertical',seed=SEED),
            tf.keras.layers.RandomRotation(.05,fill_mode='reflect',seed=SEED+1),
            tf.keras.layers.RandomZoom(.10,fill_mode='reflect',seed=SEED+2),
            tf.keras.layers.RandomBrightness(.10,value_range=(0,255),seed=SEED+3)])
        def prepare(images,labels):
            images=tf.clip_by_value(augmentation(images,training=True),0.,255.)
            return tf.keras.applications.resnet50.preprocess_input(images),tf.one_hot(labels,3)
    else:
        def prepare(images,labels):
            return tf.keras.applications.resnet50.preprocess_input(images),tf.one_hot(labels,3)
    return ds.map(prepare,num_parallel_calls=1,deterministic=True).prefetch(2)
TRAIN_DS=image_dataset(train_frame,training=True)
VAL_DS={name:image_dataset(frame) for name,frame in val_frames.items()}
VAL_LABELS={name:frame.label_index.to_numpy(dtype=np.int32) for name,frame in val_frames.items()}
print('Training and both validation pipelines ready; no test pipeline created.')

## Establish the starting checkpoint and save validation reports
The unchanged starting model is evaluated on both validation sets. This establishes a fair reference before the MILK training updates. Save a checkpoint only when **original validation macro F1** improves. SCC metrics are reported explicitly; accuracy alone is not used for selection.

In [ ]:
def score_table(labels,probabilities):
    probabilities=np.asarray(probabilities)
    if probabilities.shape!=(len(labels),3) or not np.isfinite(probabilities).all():raise ValueError('Invalid model scores.')
    if (probabilities<0).any() or (probabilities>1).any() or not np.allclose(probabilities.sum(axis=1),1,atol=1e-4):raise ValueError('Expected three probability scores per image.')
    predicted=probabilities.argmax(axis=1)
    report=classification_report(labels,predicted,labels=[0,1,2],target_names=CLASS_NAMES,output_dict=True,zero_division=0)
    table=pd.DataFrame([{'class_label':label,**report[label]} for label in CLASS_NAMES])
    return {'macro_f1':float(table['f1-score'].mean()),'accuracy':float((predicted==labels).mean()),
            'SCC_precision':float(table.iloc[2]['precision']),'SCC_recall':float(table.iloc[2]['recall']),
            'SCC_f1':float(table.iloc[2]['f1-score'])},table,predicted

def evaluate_and_save(model,tag):
    results={}
    for name,ds in VAL_DS.items():
        scores=model.predict(ds,verbose=0)
        metrics,table,predicted=score_table(VAL_LABELS[name],scores)
        table.to_csv(RUN_REPORTS/f'{tag}_{name}_validation_per_class_metrics.csv',index=False)
        predictions=val_frames[name].drop(columns=['local_path'],errors='ignore').copy()
        predictions['predicted_class']=[CLASS_NAMES[i] for i in predicted]
        for i,label in enumerate(CLASS_NAMES):predictions['score_'+label]=scores[:,i]
        predictions.to_csv(RUN_REPORTS/f'{tag}_{name}_validation_predictions.csv',index=False)
        predictions.loc[predictions.class_label!=predictions.predicted_class].to_csv(RUN_REPORTS/f'{tag}_{name}_validation_errors.csv',index=False)
        pd.DataFrame(confusion_matrix(VAL_LABELS[name],predicted,labels=[0,1,2]),index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(RUN_REPORTS/f'{tag}_{name}_validation_confusion_matrix.csv')
        results[name]=metrics
        print(tag,name,metrics)
        display(table)
    return results

tf.keras.backend.clear_session();gc.collect();tf.keras.utils.set_random_seed(SEED)
model=tf.keras.models.load_model(SOURCE_MODEL,compile=False,safe_mode=True)
if model.input_shape[1:]!=(*IMAGE_SIZE,3) or model.output_shape[-1]!=3:raise ValueError('Unexpected source model shape.')
bases=[layer for layer in model.layers if isinstance(layer,tf.keras.Model) and layer.name.lower().startswith('resnet50')]
if len(bases)!=1:raise ValueError('Could not uniquely identify the ResNet50 backbone.')
base=bases[0]
START_RESULTS=evaluate_and_save(model,'starting')
np.testing.assert_allclose(START_RESULTS['original']['macro_f1'],source_metadata['validation_macro_f1'],rtol=0,atol=1e-7)
np.testing.assert_allclose(START_RESULTS['original']['accuracy'],source_metadata['validation_accuracy'],rtol=0,atol=1e-7)
print('Saved-model validation reference: passed')
# Preserve the baseline as a valid fallback if every training epoch is worse.
shutil.copyfile(SOURCE_MODEL,RUN_MODELS/'best.keras')
model.save_weights(RUN_MODELS/'best.weights.h5')
write_json(RUN_REPORTS/'starting_validation_summary.json',START_RESULTS)

## Train and fine-tune
First adapt the existing head with the backbone frozen. Then unfreeze the upper half of the backbone with batch-normalisation layers frozen and a lower learning rate. Use training-only inverse-frequency class weights through weighted cross-entropy; do not also pass `class_weight` or oversample classes. Best checkpoints and epoch history are saved to Drive.

A disconnected/incomplete run is preserved under an interrupted directory when setup is rerun. A completed run is protected against overwrite. This is not an exact optimiser-state resume.

In [ ]:
class DualValidationCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best=START_RESULTS['original']['macro_f1']
        self.best_results=START_RESULTS
        self.best_phase='starting_checkpoint'
        self.best_epoch=0
        self.phase='head_adaptation'
        self.rows=[]
    def on_epoch_end(self,epoch,logs=None):
        logs=logs if logs is not None else {}
        results={}
        for name,ds in VAL_DS.items():
            scores=self.model.predict(ds,verbose=0)
            metrics,_,_=score_table(VAL_LABELS[name],scores)
            results[name]=metrics
        value=results['original']['macro_f1']
        logs['val_macro_f1']=value
        logs['milk_val_macro_f1']=results['MILK10k']['macro_f1']
        row={'phase':self.phase,'epoch':epoch+1,**{k:float(v) for k,v in logs.items()}}
        for name,metrics in results.items():
            row.update({name+'_'+key:val for key,val in metrics.items()})
        self.rows.append(row)
        pd.DataFrame(self.rows).to_csv(RUN_REPORTS/'training_history.csv',index=False)
        print(f" — original macro F1: {value:.4f}; SCC recall: {results['original']['SCC_recall']:.4f}; MILK macro F1: {results['MILK10k']['macro_f1']:.4f}")
        if value>self.best+1e-9:
            self.best=value;self.best_results=results;self.best_phase=self.phase;self.best_epoch=epoch+1
            temporary=RUN_MODELS/'checkpoint_pending.keras'
            self.model.save(temporary);temporary.replace(RUN_MODELS/'best.keras')
            self.model.save_weights(RUN_MODELS/'best.weights.h5')
            write_json(RUN_MODELS/'checkpoint.json',{'phase':self.phase,'epoch':epoch+1,'selection_metric':'original_validation_macro_f1','validation':results})
            print('Improved checkpoint saved to Drive.')

def compile_for_training(learning_rate):
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss=tf.keras.losses.CategoricalFocalCrossentropy(alpha=ALPHA,gamma=0.),
                  metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')])

def callbacks():
    return [monitor,
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_macro_f1',mode='max',factor=.5,patience=2,min_lr=1e-7,verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_macro_f1',mode='max',patience=5,restore_best_weights=False)]

monitor=DualValidationCheckpoint()
started=time.time()
model.trainable=True;base.trainable=False
compile_for_training(HEAD_LR)
print('HEAD ADAPTATION: ResNet50')
model.fit(TRAIN_DS,validation_data=VAL_DS['original'],epochs=HEAD_EPOCHS,callbacks=callbacks(),verbose=2)
model.load_weights(RUN_MODELS/'best.weights.h5')
base.trainable=True
cutoff=int(len(base.layers)*.50)
for index,layer in enumerate(base.layers):
    layer.trainable=index>=cutoff and not isinstance(layer,tf.keras.layers.BatchNormalization)
monitor.phase='fine_tune'
compile_for_training(FINE_LR)
print('FINE-TUNING: ResNet50')
model.fit(TRAIN_DS,validation_data=VAL_DS['original'],epochs=FINE_TUNE_EPOCHS,callbacks=callbacks(),verbose=2)
training_seconds=time.time()-started
print('Training phases finished. Best phase:',monitor.best_phase)

## Verify serialization and record actual results
Reload the selected model; compare it with the corresponding weight checkpoint; save per-class reports, predictions, errors and confusion matrices separately for each validation set. No test metrics or claims of meeting test targets are produced.

In [ ]:
model.load_weights(RUN_MODELS/'best.weights.h5')
reference_batch,_=next(iter(VAL_DS['original']))
reference_scores=model.predict(reference_batch,verbose=0)
loaded=tf.keras.models.load_model(RUN_MODELS/'best.keras',compile=False,safe_mode=True)
reloaded_scores=loaded.predict(reference_batch,verbose=0)
np.testing.assert_allclose(reference_scores,reloaded_scores,rtol=1e-4,atol=1e-5)
np.savez(RUN_REPORTS/'validation_reference.npz',scores=reloaded_scores,labels=VAL_LABELS['original'][:len(reloaded_scores)])
original_val.head(len(reloaded_scores)).drop(columns=['local_path'],errors='ignore').to_csv(RUN_REPORTS/'validation_reference_images.csv',index=False)
FINAL_RESULTS=evaluate_and_save(loaded,'best')
np.testing.assert_allclose(FINAL_RESULTS['original']['macro_f1'],monitor.best,atol=1e-7,rtol=0)
comparison=pd.DataFrame([{'validation_set':name,'checkpoint':stage,**results[name]}
                        for stage,results in [('starting',START_RESULTS),('best',FINAL_RESULTS)] for name in val_frames])
comparison.to_csv(RUN_REPORTS/'validation_comparison.csv',index=False)
display(comparison)
improved=FINAL_RESULTS['original']['macro_f1']>START_RESULTS['original']['macro_f1']+1e-9
metadata={'model_name':'ResNet50','run_id':RUN_ID,'training_complete':True,
          'model_file':str(RUN_MODELS/'best.keras'),'model_sha256':sha256_file(RUN_MODELS/'best.keras'),
          'class_names':CLASS_NAMES,'class_to_index':CLASS_TO_INDEX,'image_size':list(IMAGE_SIZE),
          'color_mode':'RGB','resize':'tf.image.resize bilinear; stretch to 224x224',
          'decode':'tf.io.decode_image channels=3; no EXIF orientation transform',
          'preprocessing':'resnet50','preprocessing_location':'outside saved model',
          'tensorflow_version':tf.__version__,'keras_version':getattr(tf.keras,'__version__',None),
          'manifest_hashes':original_hashes,'experiment_settings':SETTINGS,
          'selection_metric':'original_validation_macro_f1',
          'validation_macro_f1':FINAL_RESULTS['original']['macro_f1'],
          'validation_accuracy':FINAL_RESULTS['original']['accuracy'],
          'validation_results_by_source':FINAL_RESULTS,'starting_validation_results':START_RESULTS,
          'improved_original_validation_macro_f1':improved,
          'best_phase':monitor.best_phase,'best_epoch':monitor.best_epoch,
          'training_seconds':round(training_seconds,2),'created_at_utc':datetime.now(timezone.utc).isoformat(),
          'original_test_results_already_inspected':True,'test_inference_performed_in_this_notebook':False,
          'test_images_used_for_gradient_updates':False,'synthetic_images_used':False,
          'darker_skin_reliability_established':False}
write_json(RUN_MODELS/'metadata.json',metadata)
history=pd.DataFrame(monitor.rows)
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(history['loss'],label='training loss');axes[0].plot(history['val_loss'],label='original validation loss')
axes[1].plot(history['val_macro_f1'],label='original validation macro F1')
axes[1].plot(history['milk_val_macro_f1'],label='MILK validation macro F1')
for axis in axes:axis.set_xlabel('Recorded epoch (head, then fine-tuning)');axis.legend()
fig.tight_layout();fig.savefig(RUN_REPORTS/'training_curves.png',dpi=160);plt.show()

orig=FINAL_RESULTS['original']; milk=FINAL_RESULTS['MILK10k']
summary=(f"## Real MILK data training results — {RUN_ID}\n\n"
         f"Training images: {len(train_frame)}; SCC training images: {int(train_counts[2])}.\n\n"
         f"Original validation: accuracy {orig['accuracy']:.4f}, macro F1 {orig['macro_f1']:.4f}, "
         f"SCC precision {orig['SCC_precision']:.4f}, recall {orig['SCC_recall']:.4f}, F1 {orig['SCC_f1']:.4f}.\n\n"
         f"MILK validation: accuracy {milk['accuracy']:.4f}, macro F1 {milk['macro_f1']:.4f}, "
         f"SCC precision {milk['SCC_precision']:.4f}, recall {milk['SCC_recall']:.4f}, F1 {milk['SCC_f1']:.4f}.\n\n"
         f"Original validation macro F1 improved over the starting model: {improved}. Best phase: {monitor.best_phase}.\n\n"
         "Test targets have not been assessed for this checkpoint. Darker-skin reliability remains unestablished. "
         "MILK tone and Fitzpatrick annotations are different systems. No app model was replaced.")
(RUN_REPORTS/'results_summary.md').write_text(summary,encoding='utf-8')
print('REAL MILK TRAINING COMPLETE')
print('Run ID:',RUN_ID)
print('Original validation macro F1 improved:',improved)
print('Selected checkpoint phase:',monitor.best_phase)
print('Model:',RUN_MODELS/'best.keras')
print('Reports:',RUN_REPORTS)
print('No test evaluation or app model replacement performed.')
print('Copy the following into the final Text cell if desired:')
print(summary)

## Record the outcome
After successful completion, paste the printed results summary into a Text cell here. Report actual original and MILK validation scores; do not claim test targets or darker-skin reliability from these results. If `starting_checkpoint` remains the best phase, the training updates did not improve the primary validation metric.

Save to Drive and GitHub as `notebooks/13_milk10k_model_training.ipynb`.
Commit message: `Record real MILK data training and separate validation results`.

Keep the current Streamlit model until a selected completed candidate has been evaluated, exported and checked for inference parity. Do not copy these model weights or private validation reference images into GitHub.